# Data Cleaning Notebook
Week 4 IDX Exchange


Jack Phelan

# Setup

In [158]:
#imports
import pandas as pd
import re
from geopy.geocoders import Nominatim
from tqdm import tqdm
from geopy.extra.rate_limiter import RateLimiter
from geopy.geocoders import Nominatim

In [159]:
# read in data
enriched_listings = pd.read_csv("./../data/enriched/listings_with_rates.csv", low_memory=False)
enriched_sold = pd.read_csv("./../data/enriched/sold_with_rates.csv", low_memory=False)

# Datetime Conversions

In [160]:
def dt_converter(df): 
    # Convert date fields to datetime format
    date_fields = ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']
    for field in date_fields:
        if field in df.columns:
            df[field] = pd.to_datetime(df[field], errors='coerce')
    return df

In [161]:
enriched_listings.head(5)

,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress.1,BuyerAgencyCompensationType,BuyerAgencyCompensation,year_month
0,929000.0,1076194146,dianne@drector.com,NaN,NaN,Dianne,Rector,NaN,NaN,16882 Canyon Lane,...,Huntington Beach Union High,92649,NaN,330.0,1847.0,NaN,16882 Canyon Lane,NaN,NaN,2024-05
1,999999.0,1076194026,realestateby_denisegarcia@gmail.com,NaN,NaN,Denise,Garcia,NaN,NaN,8720 S 4th Avenue,...,Inglewood Unified,90305,NaN,0.0,8508.0,NaN,8720 S 4th Avenue,NaN,NaN,2024-05
2,1400000.0,1076193814,alizabethjames@hotmail.com,NaN,NaN,Alizabeth,James,33.858559,-116.542169,505 E Molino Road,...,Palm Springs Unified,92262,NaN,NaN,10890.0,NaN,505 E Molino Road,NaN,NaN,2024-05
3,4998888.0,1076193812,ernieramos62@yahoo.com,NaN,NaN,Ernesto,Ramos,NaN,NaN,3653 Halldale Avenue,...,NaN,90018,NaN,NaN,6192.0,NaN,3653 Halldale Avenue,NaN,NaN,2024-05
4,549000.0,1076193525,parsanina@yahoo.com,NaN,NaN,Nina,Parsa,NaN,NaN,1736 N Mcdivitt Avenue,...,Los Angeles Unified,90221,NaN,0.0,4113.0,NaN,1736 N Mcdivitt Avenue,NaN,NaN,2024-05


In [162]:
enriched_listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 893594 entries, 0 to 893593
Data columns (total 85 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   OriginalListPrice             890150 non-null  float64
 1   ListingKey                    893594 non-null  int64  
 2   ListAgentEmail                891222 non-null  object 
 3   CloseDate                     257985 non-null  object 
 4   ClosePrice                    233372 non-null  float64
 5   ListAgentFirstName            888314 non-null  object 
 6   ListAgentLastName             893511 non-null  object 
 7   Latitude                      781122 non-null  float64
 8   Longitude                     781848 non-null  float64
 9   UnparsedAddress               891239 non-null  object 
 10  PropertyType                  893594 non-null  object 
 11  LivingArea                    782759 non-null  float64
 12  ListPrice                     891376 non-nul

## Task List
- Convert date fields to datetime format (CloseDate, PurchaseContractDate, ListingContractDate, ContractStatusChangeDate) OK
- Remove unnecessary or redundant columns
- Handle missing values appropriately
- Ensure numeric fields are properly typed
- Remove or flag invalid numeric values: ClosePrice <= 0, LivingArea <= 0, DaysOnMarket < 0,
negative Bedrooms or Bathrooms

In [163]:
int_list = dt_converter(enriched_listings)

In [164]:
dt_cols = ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']
int_list[dt_cols].head(5)

,CloseDate,PurchaseContractDate,ListingContractDate,ContractStatusChangeDate
0,NaT,NaT,2024-05-10,2024-05-10
1,NaT,NaT,2024-05-30,2024-05-30
2,NaT,NaT,2024-05-21,2024-05-21
3,NaT,NaT,2024-05-29,2024-05-29
4,NaT,NaT,2024-05-27,2024-05-27


# Removing Unnecessary Columns

In [165]:
cols = [
    "AboveGradeFinishedArea",
    "BelowGradeFinishedArea",
    "TaxAnnualAmount",
    "BuilderName",
    "ElementarySchoolDistrict",
    "BusinessType",
    "CoveredSpaces",
    "MiddleOrJuniorSchool",
    "MiddleOrJuniorSchoolDistrict",
    "Latitude.1",
    "Longitude.1",
    "UnparsedAddress.1",
    "FireplacesTotal",
    "DaysOnMarket.1",
    "LivingArea.1",
    "ListPrice.1",
    "CloseDate.1",
    "BuyerOfficeName.1",
    "PropertyType.1",
    "HighSchool",
    "ElementarySchool"
]

In [166]:
int_list = int_list.drop(columns=cols,axis=1)

In [167]:
int_list.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 893594 entries, 0 to 893593
Data columns (total 64 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   OriginalListPrice            890150 non-null  float64       
 1   ListingKey                   893594 non-null  int64         
 2   ListAgentEmail               891222 non-null  object        
 3   CloseDate                    257985 non-null  datetime64[ns]
 4   ClosePrice                   233372 non-null  float64       
 5   ListAgentFirstName           888314 non-null  object        
 6   ListAgentLastName            893511 non-null  object        
 7   Latitude                     781122 non-null  float64       
 8   Longitude                    781848 non-null  float64       
 9   UnparsedAddress              891239 non-null  object        
 10  PropertyType                 893594 non-null  object        
 11  LivingArea                

In [168]:
agent_cols = [int for int in int_list.columns if re.search(r'Agent', int)]

agent_test = ['ListAgentFirstName', 'ListAgentLastName']
agent_test_1 = ['ListAgentFirstName.1', 'ListAgentLastName.1']
agent_test_full = agent_test + agent_test_1

first = ['ListAgentFirstName', 'ListAgentFirstName.1']
last = ['ListAgentLastName', 'ListAgentLastName.1']

In [169]:
int_list[agent_test_full].head(5)

,ListAgentFirstName,ListAgentLastName,ListAgentFirstName.1,ListAgentLastName.1
0,Dianne,Rector,Dianne,Rector
1,Denise,Garcia,Denise,Garcia
2,Alizabeth,James,Alizabeth,James
3,Ernesto,Ramos,Ernesto,Ramos
4,Nina,Parsa,Nina,Parsa


In [170]:
agent_drops = [
    "ListAgentEmail",
    "ListAgentFirstName",
    "ListAgentLastName",
    "ListAgentFirstName.1",
    "ListAgentLastName.1",
]

In [171]:
int_list = int_list.drop(columns=agent_drops)

In [172]:
int_list.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 893594 entries, 0 to 893593
Data columns (total 59 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   OriginalListPrice            890150 non-null  float64       
 1   ListingKey                   893594 non-null  int64         
 2   CloseDate                    257985 non-null  datetime64[ns]
 3   ClosePrice                   233372 non-null  float64       
 4   Latitude                     781122 non-null  float64       
 5   Longitude                    781848 non-null  float64       
 6   UnparsedAddress              891239 non-null  object        
 7   PropertyType                 893594 non-null  object        
 8   LivingArea                   782759 non-null  float64       
 9   ListPrice                    891376 non-null  float64       
 10  DaysOnMarket                 893594 non-null  int64         
 11  ListOfficeName            

# Numeric Type Checking

In [173]:
int_list = int_list.copy()

In [174]:
parking_vals = int_list['ParkingTotal'].unique()

display(parking_vals) # are half parking spaces valid? idk

array([ 2.000000e+00,  0.000000e+00,  3.000000e+00,  1.000000e+00,
        4.000000e+00,           nan,  6.000000e+00,  8.000000e+00,
        9.000000e+00,  5.000000e+00,  1.000000e+01,  1.200000e+01,
        7.000000e+00,  1.800000e+01,  8.400000e+01,  2.000000e+01,
        2.200000e+01,  6.500000e+00,  2.400000e+01,  1.400000e+01,
        1.700000e+01,  3.000000e+01,  2.300000e+01,  1.100000e+01,
        2.500000e+00,  5.500000e+00,  1.300000e+01,  1.600000e+01,
        3.780000e+02,  5.240000e+02,  1.500000e+00,  1.500000e+01,
        1.000000e+03, -2.000000e+00,  2.500000e+01,  2.600000e+01,
        2.100000e+01,  1.340000e+02,  1.430000e+02,  3.300000e+01,
        4.000000e+01,  1.900000e+01,  5.000000e+01,  2.900000e+01,
        2.700000e+01,  4.500000e+00,  4.800000e+01,  7.800000e+02,
        1.000000e+02,  4.000000e+02,  3.600000e+01,  8.200000e+01,
        3.100000e+01,  6.000000e+01,  3.700000e+01,  3.200000e+01,
        4.900000e+01,  4.500000e+02, -7.000000e+00,  7.300000e

In [ ]:
to_int_cols = [
    "YearBuilt",
    "StreetNumberNumeric",
    "BathroomsTotalInteger",
    "TaxYear",
    "Stories",
]

int_list[to_int_cols] = int_list[to_int_cols].astype("Int64")

In [ ]:
int_list.head()

,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,PropertyType,LivingArea,ListPrice,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,BuyerAgencyCompensationType,BuyerAgencyCompensation,year_month
0,929000.0,1076194146,NaT,NaN,NaN,NaN,16882 Canyon Lane,Residential,1389.0,929000.0,...,2.0,False,2.0,Huntington Beach Union High,92649,330.0,1847.0,NaN,NaN,2024-05
1,999999.0,1076194026,NaT,NaN,NaN,NaN,8720 S 4th Avenue,Residential,2526.0,999999.0,...,0.0,False,2.0,Inglewood Unified,90305,0.0,8508.0,NaN,NaN,2024-05
2,1400000.0,1076193814,NaT,NaN,33.858559,-116.542169,505 E Molino Road,Residential,2256.0,1400000.0,...,NaN,False,2.0,Palm Springs Unified,92262,NaN,10890.0,NaN,NaN,2024-05
3,4998888.0,1076193812,NaT,NaN,NaN,NaN,3653 Halldale Avenue,ResidentialIncome,NaN,4998888.0,...,NaN,True,0.0,NaN,90018,NaN,6192.0,NaN,NaN,2024-05
4,549000.0,1076193525,NaT,NaN,NaN,NaN,1736 N Mcdivitt Avenue,Residential,986.0,549000.0,...,2.0,False,2.0,Los Angeles Unified,90221,0.0,4113.0,NaN,NaN,2024-05


# Missing Value Handling

Imputation strategies:
- Impute geographic location with knn of similar geography things
- then perhaps groupwise imputation for numeric values based on property type and location and maybe listing price range

## Geographic imputation

In [ ]:
geo_cols = [
    "Latitude",
    "Longitude",
    "UnparsedAddress",
    "MLSAreaMajor",
    "CountyOrParish",
    "City",
    "StateOrProvince",
    "PostalCode",
]


In [ ]:
int_list[geo_cols].head(5)

,Latitude,Longitude,UnparsedAddress,MLSAreaMajor,CountyOrParish,City,StateOrProvince,PostalCode
0,NaN,NaN,16882 Canyon Lane,17 - Northwest Huntington Beach,Orange,Huntington Beach,CA,92649
1,NaN,NaN,8720 S 4th Avenue,699 - Not Defined,Los Angeles,Inglewood,CA,90305
2,33.858559,-116.542169,505 E Molino Road,331 - North End Palm Springs,Riverside,Palm Springs,CA,92262
3,NaN,NaN,3653 Halldale Avenue,C42 - Downtown L.A.,Los Angeles,Los Angeles,CA,90018
4,NaN,NaN,1736 N Mcdivitt Avenue,"RN - Compton N of Rosecrans, E of Central",Los Angeles,Compton,CA,90221


## Imputing elsewhere
- price range
- county/zip or city
- year built maybe
- floors, area quartiles
( lots of options figure it out at some point)

In [ ]:
prices = int_list["OriginalListPrice"].quantile([0.25, 0.5, 0.75])

int_list["PriceRange"] = pd.cut(int_list["OriginalListPrice"], bins=[0, prices[0.25], prices[0.5], prices[0.75], float('inf')], labels=["Low", "Medium", "High", "Very High"])

In [ ]:
# columns that have data that wouldn't be used in predictive modeling assumingly
non_predictive_columns =   [
        "ListingKey",
        "CloseDate",
        "ClosePrice",
        "Latitude",
        "Longitude",
        "UnparsedAddress",
        "ListOfficeName",
        "BuyerOfficeName",
        "CoListOfficeName",
        "ListAgentFullName",
        "CoListAgentFirstName",
        "CoListAgentLastName",
        "BuyerAgentMlsId",
        "BuyerAgentFirstName",
        "BuyerAgentLastName",
        "ListingKeyNumeric",
        "SubdivisionName",
        "BuyerOfficeAOR",
        "StreetNumberNumeric",
        "ListingId",
        "ContractStatusChangeDate",
        "CoBuyerAgentFirstName",
        "PurchaseContractDate",
        "ListingContractDate",
        "LotSizeDimensions",
        "BuyerAgencyCompensationType",
        "BuyerAgencyCompensation",
    ]


In [ ]:
pr_list = int_list.drop(columns=non_predictive_columns, axis=1)

pr_list.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 893594 entries, 0 to 893593
Data columns (total 33 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   OriginalListPrice        890150 non-null  float64 
 1   PropertyType             893594 non-null  object  
 2   LivingArea               782759 non-null  float64 
 3   ListPrice                891376 non-null  float64 
 4   DaysOnMarket             893594 non-null  int64   
 5   AssociationFeeFrequency  258896 non-null  object  
 6   MLSAreaMajor             792414 non-null  object  
 7   CountyOrParish           893593 non-null  object  
 8   MlsStatus                893594 non-null  object  
 9   AttachedGarageYN         589448 non-null  object  
 10  ParkingTotal             839102 non-null  float64 
 11  PropertySubType          785189 non-null  object  
 12  LotSizeAcres             808569 non-null  float64 
 13  YearBuilt                821420 non-null  In

In [ ]:
pr_list.head()

,OriginalListPrice,PropertyType,LivingArea,ListPrice,DaysOnMarket,AssociationFeeFrequency,MLSAreaMajor,CountyOrParish,MlsStatus,AttachedGarageYN,...,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,year_month,PriceRange
0,929000.0,Residential,1389.0,929000.0,0,Monthly,17 - Northwest Huntington Beach,Orange,Active,True,...,1847.0,2.0,False,2.0,Huntington Beach Union High,92649,330.0,1847.0,2024-05,High
1,999999.0,Residential,2526.0,999999.0,0,NaN,699 - Not Defined,Los Angeles,Active,False,...,8508.0,0.0,False,2.0,Inglewood Unified,90305,0.0,8508.0,2024-05,High
2,1400000.0,Residential,2256.0,1400000.0,16,NaN,331 - North End Palm Springs,Riverside,Active,True,...,10890.0,NaN,False,2.0,Palm Springs Unified,92262,NaN,10890.0,2024-05,Very High
3,4998888.0,ResidentialIncome,NaN,4998888.0,0,NaN,C42 - Downtown L.A.,Los Angeles,Active,NaN,...,6192.0,NaN,True,0.0,NaN,90018,NaN,6192.0,2024-05,Very High
4,549000.0,Residential,986.0,549000.0,0,NaN,"RN - Compton N of Rosecrans, E of Central",Los Angeles,Active,False,...,4113.0,2.0,False,2.0,Los Angeles Unified,90221,0.0,4113.0,2024-05,Medium


Imputation strategies:
- default = groupwise imputation
- geodata check others (zip, city, etc) 
- remove impossible values

In [ ]:
[
    "HighSchoolDistrict", # 35,4786 missing rows
    "PostalCode", #218 missing rows
    "City",   #977 missing rows
    "MLSAreaMajor", # 10,1180 missing (11%)
    "CountyOrParish" # 1 missing
]

['HighSchoolDistrict', 'PostalCode', 'City', 'MLSAreaMajor', 'CountyOrParish']

In [ ]:
# flagging bad values 

non_neg_cols = [
        "OriginalListPrice",
        "LivingArea",
        "ListPrice",
        "DaysOnMarket",
        "ParkingTotal",
        "LotSizeAcres",
        "BathroomsTotalInteger",
        "BuildingAreaTotal",
        "BedroomsTotal",
        "LotSizeArea",
        "MainLevelBedrooms",
        "GarageSpaces",
        "LotSizeSquareFeet",
    ]

neg_check = int_list[non_neg_cols] < 0
int_list["impossible_values_flag"] = neg_check.any(axis=1)

year_check = int_list["YearBuilt"] > 2026
int_list["impossible_year_flag"] = year_check


In [ ]:
# lat and long checking
int_list["null_coords_flag"] = int_list[["Latitude", "Longitude"]].isnull().any(axis=1)
int_list["placeholder_coords_flag"] = (int_list["Latitude"] == 0) | (int_list["Longitude"] == 0)
int_list["non_cali_coords_flag"] = (int_list["Latitude"] > 32) & (int_list["Latitude"] < 42) & (int_list["Longitude"] > -124) & (int_list["Longitude"] < -114)


In [ ]:
# validating order of date columns

int_list["listing_after_close_flag"] = int_list["ListingContractDate"] > int_list["CloseDate"]
int_list["purchase_after_close_flag"] = int_list["PurchaseContractDate"] > int_list["CloseDate"]
int_list["negative_timeline_flag"] = int_list["ListingContractDate"] > int_list["PurchaseContractDate"]